# ANN Classroom Demo — Breast Cancer Classification with PyTorch


### Cell 1 — Install/Import Libraries

In [ ]:
# Cell 1: Install/import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# For reproducible results in front of a class
torch.manual_seed(42)
np.random.seed(42)

print("Libraries imported successfully. PyTorch version:", torch.__version__)

### Cell 2 — Load Dataset

In [ ]:
# Cell 2: Load dataset
# We use the Breast Cancer Wisconsin dataset built into scikit-learn.
# It is small, clean, and needs no downloads or file uploads.

data = load_breast_cancer()

# data.data  -> feature values (30 numeric features)
# data.target -> labels (0 = malignant, 1 = benign)
print("Dataset loaded.")
print("Feature names (first 5):", data.feature_names[:5])
print("Target names:", data.target_names)

### Cell 3 — Explore the Dataset

In [ ]:
# Cell 3: Explore the dataset

df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

print("Shape of dataset:", df.shape)
print()
print("First 5 rows:")
display(df.head())

print()
print("Class distribution (0 = malignant, 1 = benign):")
print(df['target'].value_counts())

print()
print("Basic statistics:")
display(df.describe())

### Cell 4 — Prepare X and y

In [ ]:
# Cell 4: Prepare X and y

X = data.data          # shape: (n_samples, 30) -> input features
y = data.target        # shape: (n_samples,)    -> labels (0 or 1)

print("X shape:", X.shape)
print("y shape:", y.shape)

### Cell 5 — Train/Test Split

In [ ]:
# Cell 5: Train/test split
# We keep 80% of the data for training and 20% for testing.
# stratify=y keeps the same class balance in both sets.

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

### Cell 6 — Feature Scaling

In [ ]:
# Cell 6: Feature scaling using StandardScaler
# Neural networks train much better when input features are on a similar scale.
# IMPORTANT: fit the scaler ONLY on training data, then apply it to test data.
# This avoids "data leakage" (see the Common Mistakes section).

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # learn mean/std from train, then scale train
X_test_scaled = scaler.transform(X_test)         # reuse the SAME mean/std to scale test

print("Mean of first scaled training feature (should be ~0):", X_train_scaled[:, 0].mean())
print("Std of first scaled training feature (should be ~1):", X_train_scaled[:, 0].std())

### Cell 7 — Convert Data to PyTorch Tensors

In [ ]:
# Cell 7: Convert data to PyTorch tensors
# PyTorch models only understand tensors, not NumPy arrays or DataFrames.

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)  # shape: (n, 1)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

print("X_train_tensor shape:", X_train_tensor.shape)
print("y_train_tensor shape:", y_train_tensor.shape)

### Cell 8 — Create DataLoader

In [ ]:
# Cell 8: Create DataLoader
# A DataLoader feeds the data to the model in small batches instead of all at once.
# This is standard practice, even for small datasets like this one.

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

print("Number of batches per epoch:", len(train_loader))

### Cell 9 — Define the ANN

In [ ]:
# Cell 9: Define the ANN using torch.nn.Module

class SimpleANN(nn.Module):
    def __init__(self, input_size):
        super(SimpleANN, self).__init__()
        self.fc1 = nn.Linear(input_size, 16)  # input layer -> hidden layer (16 neurons)
        self.relu = nn.ReLU()                 # non-linear activation
        self.fc2 = nn.Linear(16, 1)           # hidden layer -> output layer (1 neuron)
        # NOTE: we do NOT apply Sigmoid here because we will use
        # BCEWithLogitsLoss, which applies Sigmoid internally (more numerically stable).

    def forward(self, x):
        x = self.fc1(x)     # z1 = W1*x + b1
        x = self.relu(x)    # a1 = ReLU(z1)
        x = self.fc2(x)     # z2 = W2*a1 + b2  (raw "logit" output)
        return x

input_size = X_train_tensor.shape[1]  # 30 features
model = SimpleANN(input_size)
print(model)

### Cell 10 — Display Model Architecture

In [ ]:
# Cell 10: Display/print the model architecture

print(model)
print()

total_params = sum(p.numel() for p in model.parameters())
print("Total trainable parameters:", total_params)

for name, param in model.named_parameters():
    print(f"{name:15s} -> shape {tuple(param.shape)}")

### Cell 11 — Loss Function and Optimizer

In [ ]:
# Cell 11: Define loss function and optimizer

criterion = nn.BCEWithLogitsLoss()   # Binary Cross-Entropy loss (for 2-class problems)
learning_rate = 0.01
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

print("Loss function:", criterion)
print("Optimizer:", optimizer)

### Cell 12 — The Training Loop (as a Reusable Function)

In [ ]:
# Cell 12: Write the training loop
# We wrap it in a function so we can easily re-run experiments later (Cell 19).

def train_model(model, train_loader, criterion, optimizer, num_epochs):
    loss_history = []

    for epoch in range(num_epochs):
        model.train()              # put model in "training mode"
        epoch_loss = 0.0

        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()          # 1. clear old gradients

            outputs = model(X_batch)       # 2. forward pass -> predictions
            loss = criterion(outputs, y_batch)  # 3. compute the loss

            loss.backward()                # 4. backpropagation -> compute gradients
            optimizer.step()               # 5. update weights using the gradients

            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(train_loader)
        loss_history.append(avg_loss)

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}]  Loss: {avg_loss:.4f}")

    return loss_history

### Cell 13 — Train the Model and Record Loss

In [ ]:
# Cell 13: Train the model and record training loss

num_epochs = 50
loss_history = train_model(model, train_loader, criterion, optimizer, num_epochs)

### Cell 14 — Plot Training Loss vs Epochs

In [ ]:
# Cell 14: Plot training loss vs epochs

plt.figure(figsize=(7, 4))
plt.plot(range(1, len(loss_history) + 1), loss_history, marker='o', markersize=3)
plt.title("Training Loss vs Epochs")
plt.xlabel("Epoch")
plt.ylabel("Loss (BCEWithLogitsLoss)")
plt.grid(True)
plt.show()

### Cell 15 — Evaluate on the Test Dataset

In [ ]:
# Cell 15: Evaluate on the test dataset

model.eval()  # put model in "evaluation mode" (disables training-only behavior)

with torch.no_grad():  # we don't need gradients for evaluation
    test_logits = model(X_test_tensor)
    test_probs = torch.sigmoid(test_logits)          # convert logits -> probabilities (0 to 1)
    test_preds = (test_probs >= 0.5).float()         # convert probabilities -> class 0 or 1

print("Sample predicted probabilities:", test_probs[:5].squeeze().tolist())
print("Sample predicted classes:      ", test_preds[:5].squeeze().tolist())
print("Sample actual classes:         ", y_test_tensor[:5].squeeze().tolist())

### Cell 16 — Accuracy, Confusion Matrix, Classification Report

In [ ]:
# Cell 16: Calculate accuracy, confusion matrix, and classification report

y_true = y_test_tensor.numpy()
y_pred = test_preds.numpy()

acc = accuracy_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)
report = classification_report(y_true, y_pred, target_names=data.target_names)

print(f"Test Accuracy: {acc * 100:.2f}%")
print()
print("Confusion Matrix:")
print(cm)
print()
print("Classification Report:")
print(report)

### Cell 17 — Visualize the Confusion Matrix

In [ ]:
# Cell 17: Visualize the confusion matrix

plt.figure(figsize=(5, 4))
plt.imshow(cm, cmap='Blues')
plt.title("Confusion Matrix")
plt.colorbar()
plt.xticks([0, 1], data.target_names)
plt.yticks([0, 1], data.target_names)
plt.xlabel("Predicted label")
plt.ylabel("True label")

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, str(cm[i, j]), ha='center', va='center',
                  color='white' if cm[i, j] > cm.max() / 2 else 'black')

plt.tight_layout()
plt.show()

### Cell 18 — Demonstrate Predictions on a Few Test Samples

In [ ]:
# Cell 18: Demonstrate predictions on a few test samples

num_samples_to_show = 8
sample_indices = np.random.choice(len(X_test_tensor), num_samples_to_show, replace=False)

print(f"{'Sample':<8}{'Predicted':<12}{'Actual':<12}{'Probability':<12}")
for idx in sample_indices:
    pred_label = data.target_names[int(test_preds[idx].item())]
    true_label = data.target_names[int(y_test_tensor[idx].item())]
    prob = test_probs[idx].item()
    print(f"{idx:<8}{pred_label:<12}{true_label:<12}{prob:<12.3f}")

### Cell 19 — Effect of Learning Rate and Number of Epochs

In [ ]:
# Cell 19: Show how changing the learning rate or number of epochs affects training
# This re-creates a FRESH model each time so comparisons are fair.

def run_experiment(lr, epochs, label):
    exp_model = SimpleANN(input_size)
    exp_criterion = nn.BCEWithLogitsLoss()
    exp_optimizer = torch.optim.Adam(exp_model.parameters(), lr=lr)
    history = train_model(exp_model, train_loader, exp_criterion, exp_optimizer, epochs)
    plt.plot(range(1, len(history) + 1), history, label=label)

plt.figure(figsize=(8, 5))
run_experiment(lr=0.01,  epochs=50, label="lr=0.01 (baseline)")
run_experiment(lr=0.1,   epochs=50, label="lr=0.1 (higher)")
run_experiment(lr=0.001, epochs=50, label="lr=0.001 (lower)")
plt.title("Effect of Learning Rate on Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()